In [24]:
#%pip install msoffcrypto-tool
#%pip install pywin32

In [25]:
#%pip install python-calamine

In [1]:
import msoffcrypto
import pandas as pd
import io
import openpyxl
import glob
import os

In [2]:
pd.set_option('display.max_rows', None)

In [3]:
password = '18651865'
cols = 'A:H,T'
data = []

In [9]:
import os
import glob
import io
import msoffcrypto
import pandas as pd

path = r'H:\My Drive\การเงิน\ยอดขาย\[0-9][0-9][0-9][0-9]'

# 1. ค้นหาไฟล์ด้วย glob
search = os.path.join(path , "**" , 'ยอดขายรวมทุกสาขาBplus[0-9][0-9][0-9][0-9].xlsx')
files_year = glob.glob(search, recursive=True)

pattern_clean = os.path.join(path, "**", "ยอดขายรวมทุกสาขาBplus.xlsx")
files_clean = glob.glob(pattern_clean, recursive=True)

raw_excel_files = files_year + files_clean 

# --- [แก้ไขบัค 1] คัดกรองไฟล์ชั่วคราว (~$) ออกจากลิสต์ที่จะใช้งานจริง ---
excel_files = []
for file_path in raw_excel_files:
    if not os.path.basename(file_path).startswith('~$'):
        excel_files.append(file_path)

print(f"เจอไฟล์ Excel ทั้งหมดที่ใช้งานได้ {len(excel_files)} ไฟล์:")
for p in excel_files:
    print("- ", p)

# --- [แก้ไขบัค 4] ประกาศตัวแปรเก็บ DataFrame ไว้ที่นอกลูปหลัก ---
data = []

# 2. เริ่มลูปแกะไฟล์รายไฟล์
for i in excel_files:
    print(f"กำลังประมวลผลไฟล์: {os.path.basename(i)}")
    
    # --- [แก้ไขบัค 2] สร้าง BytesIO ใหม่ทุกครั้งที่เริ่มไฟล์ใหม่ ---
    temp_file = io.BytesIO()    
    
    with open(i, 'rb') as f:
        office_file = msoffcrypto.OfficeFile(f)
        
        # ใส่รหัสผ่านเพื่อปลดล็อก
        office_file.load_key(password=password)
        
        # บันทึกไฟล์ที่ปลดล็อกแล้วลงในหน่วยความจำ
        office_file.decrypt(temp_file)

        # ชี้ตำแหน่ง Pointer ของ BytesIO กลับไปที่จุดเริ่มต้นก่อนให้อ่านข้อมูล
        temp_file.seek(0)
        
        da = pd.ExcelFile(temp_file)
        
        # ปฏิเสธตัวที่ไม่เอาตรงๆ (เอาเฉพาะชีทประจำเดือน)
        month_names = [x for x in da.sheet_names if not x.startswith('รวม') and '.' in x]
        
        # ลูปอ่านแต่ละชีทในไฟล์นั้นๆ
        for m in month_names:
            df = pd.read_excel(temp_file, sheet_name=m, usecols=cols, header=4, engine='openpyxl')
            df = df.iloc[0:35, :]
            
            # แปลงข้อมูลเป็นวันที่
            df['DATE'] = pd.to_datetime(df['Unnamed: 0'], dayfirst=True, errors='coerce')
            
            # ลบค่าว่างในคอลัมน์ DATE
            df.dropna(subset=['DATE'], inplace=True)
            
            # กรองวันที่ที่มากกว่า 2020
            df = df[df['DATE'].dt.year > 2020]

            df['DATE'] = df['DATE'].dt.date
            df.drop(columns=['Unnamed: 0'], inplace=True)
            df.rename(columns={'Unnamed: 19': 'WH'}, inplace=True)

            # เก็บ DataFrame ของชีทนี้เข้าลิสต์หลัก
            data.append(df)

# --- [แก้ไขบัค 3] ย้ายการรวมร่างไฟล์ (Concat) และคลีนข้อมูลมาไว้ "นอกลูปหลัก" ---
if data:  # เช็คก่อนว่ามีข้อมูลถูกเก็บมาจริงไหม
    print("กำลังรวมข้อมูลจากทุกไฟล์และทุกชีท...")
    full = pd.concat(data, ignore_index=True)
    
    # ลบข้อมูลซ้ำ
    full.drop_duplicates(inplace=True)

    # จัดการย้ายคอลัมน์ DATE ไปไว้หน้าสุด
    if 'DATE' in full.columns:
        full.insert(0, 'DATE1', full['DATE'])
        full.drop(columns='DATE', inplace=True)
        full.rename(columns={'DATE1': 'DATE'}, inplace=True)
        
    print(f"รวมข้อมูลเสร็จสิ้น! ได้ข้อมูลทั้งหมด {len(full)} แถว")
else:
    print("ไม่พบข้อมูลที่จะนำมารวมกัน")
    full = pd.DataFrame() # ส่ง Dataframe เปล่ากลับไป

เจอไฟล์ Excel ทั้งหมดที่ใช้งานได้ 4 ไฟล์:
-  H:\My Drive\การเงิน\ยอดขาย\2567\ยอดขายรวมทุกสาขาBplus2567.xlsx
-  H:\My Drive\การเงิน\ยอดขาย\2568\ยอดขายรวมทุกสาขาBplus2568.xlsx
-  H:\My Drive\การเงิน\ยอดขาย\2569\ยอดขายรวมทุกสาขาBplus2569.xlsx
-  H:\My Drive\การเงิน\ยอดขาย\2570\ยอดขายรวมทุกสาขาBplus.xlsx
กำลังประมวลผลไฟล์: ยอดขายรวมทุกสาขาBplus2567.xlsx
กำลังประมวลผลไฟล์: ยอดขายรวมทุกสาขาBplus2568.xlsx
กำลังประมวลผลไฟล์: ยอดขายรวมทุกสาขาBplus2569.xlsx
กำลังประมวลผลไฟล์: ยอดขายรวมทุกสาขาBplus.xlsx
กำลังรวมข้อมูลจากทุกไฟล์และทุกชีท...
รวมข้อมูลเสร็จสิ้น! ได้ข้อมูลทั้งหมด 1108 แถว


In [10]:
full

,DATE,ยอดขายเค1,ยอดขายเค2,ยอดขายเค3,ยอดขายเค4,ยอดขายเค5,ยอดขายSP,ยอดขายPet,WH
0,2024-01-01,99236,49535,71070,21135,0,260529,13420,952565.45
1,2024-01-02,70414,36310,62065,17993,0,188234,21501,890942.56
2,2024-01-03,65718,28042,50346,13681,0,168520,20210,831994.10
3,2024-01-04,57866,32780,52748,13311,0,132730,23515,749366.71
4,2024-01-05,56893,36038,41021,12196,0,124224,19729,688072.57
5,2024-01-06,59240,28316,47156,14502,0,113115,18862,774135.66
6,2024-01-07,54673,35647,45253,14668,0,119675,9635,810049.82
7,2024-01-08,65253,29749,53987,16375,0,133091,26043,723226.87
8,2024-01-09,64132,26434,44954,13939,0,118505,17605,811844.15
9,2024-01-10,62613,37690,52183,15250,0,125523,20021,1013761.35


In [ ]:
path = r'H:\My Drive\การเงิน\ยอดขาย\[0-9][0-9][0-9][0-9]'


search = os.path.join(path ,"**" , 'ยอดขายรวมทุกสาขาBplus[0-9][0-9][0-9][0-9].xlsx')
files_year = glob.glob(search,recursive=True)

pattern_clean = os.path.join(path, "**", "ยอดขายรวมทุกสาขาBplus.xlsx")
files_clean = glob.glob(pattern_clean, recursive=True)

excel_files = files_year + files_clean 


print(f"เจอไฟล์ Excel ทั้งหมด {len(excel_files)} ไฟล์:")
for path in excel_files:
    print("- ", path)

for file_path in excel_files:
    # ถ้าเป็นไฟล์ชั่วคราวของ Excel ให้ข้ามไป
    if os.path.basename(file_path).startswith('~$'):
        continue
        
temp_file = io.BytesIO()    
for i in excel_files:
    with open(f'{i}', 'rb') as f:
        office_file = msoffcrypto.OfficeFile(f)
        
        # ใส่รหัสผ่านเพื่อปลดล็อก
        office_file.load_key(password=password)
        
        # บันทึกไฟล์ที่ปลดล็อกแล้วลงในหน่วยความจำ (temp_file)
        office_file.decrypt(temp_file)

        da = pd.ExcelFile(temp_file)
        # แบบที่ 1: ปฏิเสธตัวที่ไม่เอาตรงๆ
        month_names =[x for x in da.sheet_names if not x.startswith('รวม') and '.' in x]
        # 3. ใช้ Pandas อ่านไฟล์จากหน่วยความจำ
        for m in month_names:
            df = pd.read_excel(temp_file , sheet_name= m ,usecols=cols, header= 4 ,engine= 'openpyxl')
            df = df.iloc[0:35,:]
            # แปลงข้อมูลเป็นวันที่
            df['DATE'] = pd.to_datetime(df['Unnamed: 0'], dayfirst=True ,errors='coerce')
            #ลบค่าว่าง
            df.dropna(subset='DATE',inplace=True)
            #กรอกวันที่ที่มากกว่า 2020
            df = df[df['DATE'].dt.year > 2020]

            df['DATE'] = df['DATE'].dt.date

            df.drop(columns=['Unnamed: 0'] , inplace=True)
            
            df.rename(columns={'Unnamed: 19': 'WH'}, inplace=True)

            data.append(df)

        full = pd.concat(data , ignore_index=True)
        # ลบซ้ำ
        full.drop_duplicates(inplace=True)

        full.insert(0 ,'DATE1',full['DATE'])

        full.drop(columns='DATE',inplace=True)

        full.rename(columns={'DATE1':'DATE'},inplace=True)

